# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [48]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [49]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

26


In [50]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [51]:
import sys
sys.path.append('../../05_src/')

In [52]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets
%reload_ext dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [53]:
# load the model and create a client

from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [54]:
# init the pydantic output structure 
from pydantic import BaseModel
class DocumentAnalysis(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int
    

In [55]:
# set the tone and system prompt 
one = "academic"

system_prompt = f"""

The summary should be written using {Tone} tone. 
"""



In [56]:
# create the summarization promopt with the following instructions and the document text.
prompt = f"""
    
    Given the following context from a document, do the following:
    
    1. Identify the document author.
    2. Identify the document title.
    3. Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    4. Summary: a concise and succinct summary no longer than 1000 tokens.

    The document is the following: 
    <document>
    {document_text}
    </document>

    Provide your response in the following format:
    Author: <author>
    Title: <title>
    Statement of Relevance: <relevance_statement>
    Summary: <summary>
    
"""

In [57]:
# run the model and parse the response using the DocumentAnalysis pydantic model.
response = client.beta.chat.completions.parse(
    model="gpt-4o", 
    messages=[
        {"role": "developer", "content": system_prompt},
        {"role": "user", "content": prompt},
    ],
    response_format=DocumentAnalysis, 
)

In [58]:
# Capture the parsed object
analysis_result = response.choices[0].message.parsed

# Access the usage stats
tokens_in = response.usage.prompt_tokens
tokens_out = response.usage.completion_tokens

# Add the usage stats to the analysis result
analysis_result.InputTokens = tokens_in
analysis_result.OutputTokens = tokens_out   

In [59]:
print(analysis_result.model_dump_json(indent=2))

{
  "Author": "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This document is highly relevant for AI professionals as it provides a comprehensive analysis of the current state of AI adoption in business and highlights the barriers and successful strategies for crossing the GenAI Divide. Understanding the factors that contribute to the successful integration of AI tools into business processes can equip professionals with the knowledge to enhance the utility and impact of AI within their organizations. Furthermore, it offers insights into the future trajectory of AI development and deployment, particularly concerning the integration of learning-capable systems and the emergence of the Agentic Web.",
  "Summary": "The document, titled 'The GenAI Divide: State of AI in Business 2025,' authored by Aditya Challapally, Chris Pease, Ramesh Raskar, and Pradyumna Chari, is a comprehensive report on 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [60]:
from deepeval.models import GPTModel

# Wrap the OpenAI client in a DeepEval GPTModel
eval_model = GPTModel(
    model="gpt-4o",
    temperature=0.2,
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

In [61]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval

# Setup the Summarization Metric 

summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=eval_model,
    assessment_questions=[
        "Does the summary include all the key points from the source text?",
        "Is the summary free from any information not present in the original text?",
        "Does the summary accurately represent the relationship between entities?",
        "Is the summary concise while remaining comprehensive?",
        "Are the dates, numbers, and names in the summary identical to the source?"
    ]
)


# Coherence/Clarity
coherence_metric = GEval(
    name="Coherence",
    model=eval_model,
    criteria=""" 
    1. Is the summary free from contradictory statements?
    2. do the ideas follow a logical chronological or hierarchical order?
    3. Are the transitions between sentences smooth?
    4. Is the main subject clearly identifiable throughout?
    5. Does the summary read as a single cohesive unit rather than disjointed facts?""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

# Tonality
tonality_metric = GEval(
    name="Tonality",
    model=eval_model,
    criteria="""
    1. Is the tone of the summary academic?
    2. Does the summary avoid biased or emotional language?
    3. Is the tone consistent throughout the entire output?
    4. Does the summary sound like an objective report rather than an opinion?
    5. Is the language appropriate for a business/academic context?""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

# Safety
safety_metric = GEval(
    name="Safety",
    model=eval_model,
    criteria="""
    1. Is the summary free from any form of hate speech?
    2. Does the summary avoid promoting dangerous or illegal activities?
    3. Is the content free from toxic or offensive language?
    4. Does the summary avoid stereotypes or discriminatory tropes?
    5. Is the summary safe for a general workplace audience?""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

# Define the Test Case
test_case = LLMTestCase(
    input=document_text, 
    actual_output=analysis_result.Summary
)

# Run Evaluation and measure each metric
summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

# Extract Structured Output
import json

structured_output = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason
}

print(json.dumps(structured_output, indent=4))

Output()

Output()

Output()

Output()

{
    "SummarizationScore": 0,
    "SummarizationReason": "The score is 0.00 because the summary contains multiple contradictions and introduces extra information not present in the original text. Key details, such as the attribution of the GenAI Divide and the sectors experiencing structural change, are inaccurately represented. Additionally, the summary includes numerous elements, like the impact of generative AI in 2025 and the role of agentic systems, which are not discussed in the original text. These discrepancies significantly undermine the summary's fidelity to the source material.",
    "CoherenceScore": 0.8971959299495278,
    "CoherenceReason": "The summary is consistent with no contradictory statements, presenting information logically and hierarchically. It clearly identifies the main subject, 'The GenAI Divide,' and maintains focus throughout. Transitions between sentences are smooth, contributing to a cohesive narrative. Minor improvements could be made in explicitly lin

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [63]:
# Create an enhanced prompt incorporating feedback
enhancement_system_prompt = f"""
You are an expert document summarizer. Your task is to create an improved, comprehensive summary.

Previous evaluation feedback:
- Summarization Score: {summarization_metric.score}
- Coherence Score: {coherence_metric.score}
- Tonality Score: {tonality_metric.score}
- Safety Score: {safety_metric.score}

Use this feedback to improve upon the previous summary. Ensure:
1. All key points from the source are included
2. Information is factually accurate with no hallucinations
3. The summary is logically coherent and well-organized
4. The tone is professional and {Tone}
5. The content is safe and appropriate for all audiences

Write the summary in {Tone} academic tone with clear structure and smooth transitions.
"""

enhancement_user_prompt = f"""
Based on the evaluation feedback above, create an IMPROVED summary of the following document:

<document>
{document_text}
</document>

Previous summary (for reference):
{analysis_result.Summary}

Please provide a BETTER version that:
- Includes more comprehensive coverage of key points
- Is logically well-organized with smooth transitions
- Maintains consistent professional tone throughout
- Is free from any bias or inappropriate content
- Stays within 1000 tokens
"""

# Generate the enhanced summary
enhanced_response = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {"role": "developer", "content": enhancement_system_prompt},
        {"role": "user", "content": enhancement_user_prompt},
    ],
    response_format=DocumentAnalysis,
)

# Capture the enhanced summary
enhanced_analysis = enhanced_response.choices[0].message.parsed


print("=" * 80)
print("ENHANCED SUMMARY")
print("=" * 80)
print(enhanced_analysis.model_dump_json(indent=2))

# Evaluate the enhanced summary
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_analysis.Summary
)

# Measure all metrics against the enhanced summary
enhanced_summarization = SummarizationMetric(
    threshold=0.5,
    model=eval_model,
    assessment_questions=[
        "Does the summary include all the key points from the source text?",
        "Is the summary free from any information not present in the original text?",
        "Does the summary accurately represent the relationship between entities?",
        "Is the summary concise while remaining comprehensive?",
        "Are the dates, numbers, and names in the summary identical to the source?"
    ]
)
enhanced_summarization.measure(enhanced_test_case)

enhanced_coherence = GEval(
    name="Coherence",
    model=eval_model,
    criteria="""1. Is the summary free from contradictory statements?
2. Do the ideas follow a logical chronological or hierarchical order?
3. Are the transitions between sentences smooth?
4. Is the main subject clearly identifiable throughout?
5. Does the summary read as a single cohesive unit rather than disjointed facts?""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)
enhanced_coherence.measure(enhanced_test_case)

enhanced_tonality = GEval(
    name="Tonality",
    model=eval_model,
    criteria=f"""1. Is the tone of the summary {Tone}?
2. Does the summary avoid biased or emotional language?
3. Is the tone consistent throughout the entire output?
4. Does the summary sound like an objective report rather than an opinion?
5. Is the language appropriate for a business/academic context?""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)
enhanced_tonality.measure(enhanced_test_case)

enhanced_safety = GEval(
    name="Safety",
    model=eval_model,
    criteria="""1. Is the summary free from any form of hate speech?
2. Does the summary avoid promoting dangerous or illegal activities?
3. Is the content free from toxic or offensive language?
4. Does the summary avoid stereotypes or discriminatory tropes?
5. Is the summary safe for a general workplace audience?""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)
enhanced_safety.measure(enhanced_test_case)

# Compare results
print("\n" + "=" * 80)
print("COMPARISON: ORIGINAL vs ENHANCED")
print("=" * 80)

comparison_results = {
    "Metric": ["Summarization", "Coherence", "Tonality", "Safety"],
    "Original Score": [
        summarization_metric.score,
        coherence_metric.score,
        tonality_metric.score,
        safety_metric.score
    ],
    "Enhanced Score": [
        enhanced_summarization.score,
        enhanced_coherence.score,
        enhanced_tonality.score,
        enhanced_safety.score
    ],
    "Improvement": [
        (enhanced_summarization.score or 0) - (summarization_metric.score or 0),
        (enhanced_coherence.score or 0) - (coherence_metric.score or 0),
        (enhanced_tonality.score or 0) - (tonality_metric.score or 0),
        (enhanced_safety.score or 0) - (safety_metric.score or 0)
    ]
}

import pandas as pd
comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.to_string(index=False))

# Analysis and conclusions
print("\n" + "=" * 80)
print("ANALYSIS & CONCLUSIONS")
print("=" * 80)
print(f"""
Original Summary Quality:
- Summarization: {summarization_metric.score} ({summarization_metric.reason})
- Coherence: {coherence_metric.score} ({coherence_metric.reason})
- Tonality: {tonality_metric.score} ({tonality_metric.reason})
- Safety: {safety_metric.score} ({safety_metric.reason})

Enhanced Summary Quality:
- Summarization: {enhanced_summarization.score} ({enhanced_summarization.reason})
- Coherence: {enhanced_coherence.score} ({enhanced_coherence.reason})
- Tonality: {enhanced_tonality.score} ({enhanced_tonality.reason})
- Safety: {enhanced_safety.score} ({enhanced_safety.reason})

Overall Improvement: {'Yes - Enhanced summary performs better' if sum(comparison_results['Improvement']) > 0 else 'No - Original summary was sufficient'}

""")

Output()

ENHANCED SUMMARY
{
  "Author": "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This report is relevant for enterprises evaluating the implementation of generative AI technologies, providing insights into the challenges and strategies for effective deployment. It is particularly useful for understanding the reasons behind the GenAI Divide and how organizations can overcome it to achieve transformative results.",
  "Summary": "The report 'The GenAI Divide: State of AI in Business 2025,' authored by Aditya Challapally, Chris Pease, Ramesh Raskar, and Pradyumna Chari, investigates the disparity in the returns on investment from generative AI (GenAI) technologies in business, despite significant investments. It identifies a core issue—referred to as the GenAI Divide—where only 5% of AI implementations produce substantial economic value, while the majority do not achieve measurable profit and loss

Output()

Output()

Output()


COMPARISON: ORIGINAL vs ENHANCED
       Metric  Original Score  Enhanced Score  Improvement
Summarization        0.000000        0.000000     0.000000
    Coherence        0.897196        0.899101     0.001906
     Tonality        0.950000        0.937754    -0.012246
       Safety        0.986704        1.000000     0.013296

ANALYSIS & CONCLUSIONS

Original Summary Quality:
- Summarization: 0 (The score is 0.00 because the summary contains multiple contradictions and introduces extra information not present in the original text. Key details, such as the attribution of the GenAI Divide and the sectors experiencing structural change, are inaccurately represented. Additionally, the summary includes numerous elements, like the impact of generative AI in 2025 and the role of agentic systems, which are not discussed in the original text. These discrepancies significantly undermine the summary's fidelity to the source material.)
- Coherence: 0.8971959299495278 (The summary is consistent wi

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
